## **Hyperparemeter Search**

### **Data import and split**

In [9]:
# Imports
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load dataset
data = load_breast_cancer()

# Convert to DataFrame for better visualization
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Show first few samples
X.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [10]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64

In [12]:
# Split 
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Check shapes
print(f"Training set shape: {X_train.shape}, {y_train.shape}")

Training set shape: (455, 30), (455,)


### **Grid Search**

In [17]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Create pipeline: feature scaling + neural network
pipe = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        max_iter=1000, # Increase max iterations to ensure convergence
        early_stopping=True, # Enable early stopping to prevent overfitting
        validation_fraction=0.1, # Use 10% of training data for validation during early stopping
        n_iter_no_change=10, # Stop if no improvement for 10 iterations
        random_state=42
    )
)

# Define hyperparameters to search
# 5 hidden layer sizes, 2 activation functions, 3 regularization strengths = 30 combinations
# Note: The number of combinations grows exponentially with the number of hyperparameters and their values, so be cautious when expanding the grid.
param_grid = {
    'mlpclassifier__hidden_layer_sizes': [
        (50,), # 1 layer with 50 neurons
        (100,), # 1 layer with 100 neurons
        (50, 50), # 2 layers with 50 neurons each
        (100, 50), # 2 layers with 100 and 50 neurons
        (200, 100, 50) # 3 layers with 200, 100, and 50 neurons
        ],
    'mlpclassifier__activation': ['relu', 'tanh'],
    'mlpclassifier__alpha': [0.0001, 0.001, 0.01]
}

# Stratified cross-validation
# 30 combinations * 10 folds = 300 fits
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Grid search
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro', # Default is 'accuracy', but F1 macro is better for imbalanced datasets
    n_jobs=-1 # Use all available CPU cores for parallel processing
)

# Start timer
start = time.time()

# Fit model
grid.fit(X_train, y_train)

# End timer
end = time.time()

# Print results
print(f"Grid search completed in {end - start:.2f} seconds")

# Print number of iterations used by the best model
best_mlp = grid.best_estimator_.named_steps['mlpclassifier']
print(f"\n🧠 Iterations (epochs) used by best model: {best_mlp.n_iter_}")

#Print total number of models trained (combinations * folds)
total_models_trained = len(grid.cv_results_['params']) * cv.get_n_splits(X_train, y_train)
print(f"Total models trained (Grid): {total_models_trained}")

# Best model
print("Best hyperparameters found:")
print(grid.best_params_)

Grid search completed in 1.27 seconds

🧠 Iterations (epochs) used by best model: 13
Total models trained (Grid): 300
Best hyperparameters found:
{'mlpclassifier__activation': 'relu', 'mlpclassifier__alpha': 0.0001, 'mlpclassifier__hidden_layer_sizes': (200, 100, 50)}


### **Random Search**

In [18]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import loguniform

# Create pipeline: feature scaling + neural network
pipe = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        max_iter=1000, # Increase max iterations to ensure convergence
        early_stopping=True, # Enable early stopping to prevent overfitting
        validation_fraction=0.1, # Use 10% of training data for validation during early stopping
        n_iter_no_change=10, # Stop if no improvement for 10 iterations
        random_state=42
    )
)

# Define hyperparameters to search
# Similar search space as GridSearch, but sampled randomly
# Note: RandomizedSearchCV does not evaluate all combinations, only a fixed number (n_iter)
param_dist = {
    'mlpclassifier__hidden_layer_sizes': [
        (50,), # 1 layer with 50 neurons
        (100,), # 1 layer with 100 neurons
        (50, 50), # 2 layers with 50 neurons each
        (100, 50), # 2 layers with 100 and 50 neurons
        (200, 100, 50) # 3 layers with 200, 100, and 50 neurons
    ],
    'mlpclassifier__activation': ['relu', 'tanh'],
    'mlpclassifier__alpha': loguniform(1e-5, 1e-1) # Continuous distribution for regularization
}

# Stratified cross-validation
# n_iter * n_splits = total number of model fits
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Random search
random_search = RandomizedSearchCV(
    estimator=pipe,
    n_iter=20, # Number of random combinations to try
    cv=cv,
    scoring='f1_macro', # Default is 'accuracy', but F1 macro is better for imbalanced datasets
    random_state=42,
    n_jobs=-1 # Use all available CPU cores for parallel processing
)

# Start timer
start = time.perf_counter()

# Fit model
random_search.fit(X_train, y_train)

# End timer
end = time.perf_counter()

# Print results
print(f"Random search completed in {end - start:.2f} seconds")

# Print total number of models trained (combinations * folds)
n_candidates = len(random_search.cv_results_['params'])
n_splits = random_search.n_splits_
print(f"Total models trained (Random): {n_candidates * n_splits}")

# Best model
print("Best hyperparameters found:")
print(random_search.best_params_)

# Extract best model to check training iterations
best_mlp = random_search.best_estimator_.named_steps['mlpclassifier']

print(f"\nIterations (epochs) used: {best_mlp.n_iter_}")
print(f"Final loss: {best_mlp.loss_}")

TypeError: RandomizedSearchCV.__init__() missing 1 required positional argument: 'param_distributions'